# GPU image-model pilot for eFootPhy

This notebook runs the existing detector/landmark trainer on **your reviewed local dataset**. It is a bounded pilot, not an action/trajectory model or an accuracy guarantee. It has not been run on a GPU by this notebook's author. No dataset upload or remote storage connection is performed automatically.

Choose a CUDA GPU runtime first. Keep full data and reviewed match-group metadata available. Polygon detection labels must be normalized on a copy with `tools.vision_qc.normalize`; pitch keypoint semantics must be reviewed. The default refuses unverified independent evaluation. Missing body/action/identity annotations are not created from bounding boxes.


In [ ]:
import sys, subprocess
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required: stop here instead of silently running hours on CPU")
print(torch.__version__, torch.cuda.get_device_name(0))
ROOT = Path('/content/efootphy-gpu-pilot')
REVISION = '21e9c1351daf16af76860f71959c7fc83af459e0'
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/clonephy-blip/efootphy.git',str(ROOT)],check=True)
subprocess.run(['git','fetch','origin',REVISION],cwd=ROOT,check=True)
# Dedicated notebook checkout only; do not reuse a working checkout with edits.
if subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip():
    raise RuntimeError('Notebook checkout has edits; preserve them and use a fresh directory')
subprocess.run(['git','checkout','--detach',REVISION],cwd=ROOT,check=True)


In [ ]:
# Preserve the CUDA-enabled torch provided by the runtime; never install the CPU lock.
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'tools/vision_qc/requirements-qc.txt'),
                'ultralytics==8.4.193','onnx==1.23.2','onnxruntime==1.31.1'],check=True)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable after environment setup')
sys.path.insert(0,str(ROOT))


In [ ]:
PROFILE = 'field'                 # 'field' or 'landmarks'; this is not temporal training
DATA_YAML = Path('/content/reviewed-data/field/data.yaml')
EXPECTED_IMAGES = None             # Set to the known complete export count, not a sample count
GROUPS_JSON = None                 # Path to reviewed match/video groups for the entire export
RESEARCH_ONLY = False              # True is source-only research; it never proves generalization
WEIGHTS = None                     # Optional retained local checkpoint for this profile
OUT = Path('/content/efootphy-pilot-output')
EPOCHS = 10                        # Pilot budget, not a convergence claim
MAX_MINUTES = 15                   # Checked at checkpoint/epoch boundaries
if PROFILE not in {'field','landmarks'} or not DATA_YAML.is_file() or not isinstance(EXPECTED_IMAGES,int):
    raise ValueError('Fill the profile, reviewed data path and complete expected image count')
if not RESEARCH_ONLY and (GROUPS_JSON is None or not Path(GROUPS_JSON).is_file()):
    raise ValueError('Verified group split is required for independent evaluation')
if OUT.exists() and any(OUT.iterdir()):
    raise ValueError('Use a fresh output directory')


In [ ]:
from tools.vision_qc.train import preflight
qc = preflight(DATA_YAML,EXPECTED_IMAGES,Path(GROUPS_JSON) if GROUPS_JSON else None,PROFILE,RESEARCH_ONLY)
print({k:qc.get(k) for k in ['split_counts','data_valid','eligible_for_independent_evaluation']})
# Existing YOLO training fine-tunes trainable backbone/head; it does not consume a frozen feature cache.
cmd=[sys.executable,'-m','tools.vision_qc.train','--data',str(DATA_YAML),
     '--profile',PROFILE,'--expected-images',str(EXPECTED_IMAGES),'--device','0',
     '--epochs',str(EPOCHS),'--patience','5','--batch','8','--nominal-batch','32',
     '--lr','0.0005','--max-minutes',str(MAX_MINUTES),'--no-plots','--out',str(OUT)]
if GROUPS_JSON:cmd += ['--groups',str(GROUPS_JSON)]
if RESEARCH_ONLY:cmd += ['--research-only']
if WEIGHTS:
    if not Path(WEIGHTS).is_file():raise ValueError('Retained weights file missing')
    cmd += ['--weights',str(WEIGHTS)]
subprocess.run(cmd,cwd=ROOT,check=True)


## Review before replacing anything

Keep the frozen baseline. Inspect original-pixel errors, class-specific ball metrics and every held-out video/match. A lower training loss, successful ONNX export or high person AP does not establish ball, identity, action or trajectory quality. Preserve the run's manifest, dataset/group audit and best/last checkpoint evidence. The action and ball-flight branches require their own sequence labels, causal evaluation and mobile profiling.
